In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install numpy pandas scikit-learn tensorflow matplotlib seaborn shap lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=ca1e57f6c2440e43e16f6045852ab3e5abf821f94a5f68cfa20a431a6bee5360
  Stored in directory: /root/.cache/pip/wheels/85/fa/a3/9c2d44c9f3cd77cf4e533b58900b2bf4487f2a17e8ec212a3d
Successfully built lime


In [ ]:
# -*- coding: utf-8 -*-
"""
Context-Aware Quantum-Inspired Intelligence for AI-Aided Cyber Worm Detection
Simulation with LSTM, SHAP, and LIME

This Jupyter Notebook provides a simulation using an LSTM model for predicting
different network traffic types, leveraging a custom dataset from a CSV file.
It incorporates context-aware quantum-inspired intelligence by performing
feature selection on network and contextual features and analyzes performance
with and without context-aware threat analysis.  It also adds
SHAP and LIME explainability to the LSTM model's predictions.

The notebook includes:
    1. Loading and preprocessing a custom dataset, separating network and contextual features.
    2. Preparing sequential data for the LSTM model.
    3. Building and training the LSTM model for multiclass classification.
    4. Evaluating the model with accuracy, precision, recall, F1-score, and loss.
    5. Plotting accuracy and loss curves during training.
    6. Generating and visualizing the confusion matrix for both scenarios.
    7. Plotting the AUC-ROC curve for each traffic type (One-vs-Rest).
    8. Using SHAP and LIME to explain LSTM predictions.
    9. Context-aware threat analysis and performance comparison.

Assumptions about the CSV file:
    - It contains network traffic features and contextual features.
    - One column is the 'traffic_type' column with more than 2 unique traffic types.
    - Data is structured or can be structured as sequential data.
    - Non-numeric columns are converted to numeric.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import random

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Explainability
import shap
import lime
import lime.lime_tabular

# Seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)


# --- 1. Load and Preprocess Custom Dataset ---
csv_file_path = '/content/drive/MyDrive/conquest-milcom/ACI-IoT-2023.csv'  # Replace with the actual path

try:
    df = pd.read_csv(csv_file_path)
    print(f"Dataset loaded successfully from {csv_file_path}")
except FileNotFoundError:
    print(f"Error: CSV file not found at {csv_file_path}")
    exit()



# Drop an unwanted column
df_dropp = df.drop('Connection Type', axis=1)


# Align data set column for netwrok and context feature selection
col_to_move1 = 'Flow ID'
new_position = 82                       # New index (0-based)

# Reorder columns
cols = df_dropp.columns.tolist()
cols.insert(new_position, cols.pop(cols.index(col_to_move1)))
df_dropp = df_dropp[cols]



print(df_dropp.head())

# To get some basic information about the DataFrame:
print(df_dropp.info())

# To see descriptive statistics:
print(df_dropp.describe())


# Align data set column for netwrok and context feature selection
col_to_move1 = 'Timestamp'
new_position = 82                       # New index (0-based)

# Reorder columns
cols = df_dropp.columns.tolist()
cols.insert(new_position, cols.pop(cols.index(col_to_move1)))
df_dropp = df_dropp[cols]


df_dropped = df_dropp


# Identify the target column
target_column = 'Label'
if target_column not in df_dropped.columns:
    print(f"Error: Target column '{target_column}' not found.")
    exit()

y_original = df_dropped[target_column].values
X = df_dropped.drop(columns=[target_column])


# Convert all non-numeric columns to numeric
for column in X.columns:
    if pd.api.types.is_object_dtype(X[column]):
        try:
            X[column] = pd.to_numeric(X[column], errors='raise')
        except ValueError:
            print(f"Converting non-numeric column: {column}")
            le = LabelEncoder()
            X[column] = le.fit_transform(X[column])
    elif not pd.api.types.is_numeric_dtype(X[column]):
        print(f"Warning: Column '{column}' is not numeric and not string. Handling needed.")

X = X.values  # Convert features to numpy array


# --- 2. Feature Selection Preparation ---
# Separate network and contextual features.
# ... (rest of your code) ...

# Separate network and context features (you might need to adjust these based on your dataset)
# Assuming the first few columns are network features and the rest are context
n_network_features = 74  # Adjust based on your dataset

# Use standard array slicing for NumPy arrays
X_network = X[:, :n_network_features]
X_contextual = X[:, n_network_features:]

# ... (rest of your code) ...


unique_traffic_types = np.unique(y_original)
num_classes = len(unique_traffic_types)
print(f"Number of unique traffic types: {num_classes}")

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_original)
y_categorical = to_categorical(y_encoded, num_classes=num_classes)


# Convert integer labels to one-hot encoded vectors
y_categorical = to_categorical(y_encoded, num_classes=num_classes)



def simple_fitness_evaluation(feature_indices, data, labels):
    """
    Simplified fitness function (variance ratio).
    """
    if not feature_indices:
        return 0
    normal_data = data[labels == 0][:, feature_indices]
    anomalous_data = data[labels == 1][:, feature_indices]
    if normal_data.size == 0 or anomalous_data.size == 0:
        return 0
    return np.var(anomalous_data) / (np.var(normal_data) + 1e-6)

def qpso(data, labels, num_particles=10, iterations=20, num_features_to_select=10):
    """
    Simplified QPSO implementation.
    """
    n_features = data.shape[1]
    particles = [np.random.choice(n_features, num_features_to_select, replace=False) for _ in range(num_particles)]
    p_best_positions = list(particles)
    p_best_fitnesses = [simple_fitness_evaluation(p, data, labels) for p in particles]
    g_best_position = particles[np.argmax(p_best_fitnesses)]
    g_best_fitness = max(p_best_fitnesses)

    for _ in range(iterations):
        for i in range(num_particles):
            # Simplified particle update
            phi = np.random.rand()
            p = particles[i]
            x_mean = np.mean(np.array([p_best_positions[i], g_best_position]), axis=0).astype(int)
            new_position = np.zeros_like(p)
            for j in range(len(p)):
                if np.random.rand() < phi:
                    new_position[j] = x_mean[j]
                else:
                    new_position[j] = p[j]

            # Ensure no duplicate features in a particle
            new_position = np.unique(new_position)
            if len(new_position) != num_features_to_select:
                remaining_features = set(range(n_features)) - set(new_position)
                new_position = np.concatenate((new_position, np.random.choice(list(remaining_features), num_features_to_select - len(new_position), replace=False)))
            particles[i] = new_position

            # Evaluate fitness
            current_fitness = simple_fitness_evaluation(particles[i], data, labels)

            # Update p_best
            if current_fitness > p_best_fitnesses[i]:
                p_best_fitnesses[i] = current_fitness
                p_best_positions[i] = particles[i]

            # Update g_best
            if current_fitness > g_best_fitness:
                g_best_fitness = current_fitness
                g_best_position = particles[i]

    return g_best_position




def simple_fitness_evaluation(feature_indices, data, labels):
    """
    Simplified fitness function (variance ratio).
    """
    # Check if feature_indices is empty using len()
    if len(feature_indices) == 0:  # Changed this line
        return 0
    normal_data = data[labels == 0][:, feature_indices]
    anomalous_data = data[labels == 1][:, feature_indices]
    if normal_data.size == 0 or anomalous_data.size == 0:
        return 0
    return np.var(anomalous_data) / (np.var(normal_data) + 1e-6)




#New function
# --- 3. Simplified Quantum-Inspired Particle Swarm Optimization (QPSO) ---
def simple_fitness_evaluation(feature_indices, data, labels):
    """
    Simplified fitness function (variance ratio).
    """
    if not feature_indices:
        return 0
    normal_data = data[labels == 0][:, feature_indices]
    anomalous_data = data[labels == 1][:, feature_indices]
    if normal_data.size == 0 or anomalous_data.size == 0:
        return 0
    return np.var(anomalous_data) / (np.var(normal_data) + 1e-6)

def qpso(data, labels, num_particles=10, iterations=20, num_features_to_select=10):
    """
    Simplified QPSO implementation.
    """
    n_features = data.shape[1]
    particles = [np.random.choice(n_features, num_features_to_select, replace=False) for _ in range(num_particles)]
    p_best_positions = list(particles)
    p_best_fitnesses = [simple_fitness_evaluation(p, data, labels) for p in particles]
    g_best_position = particles[np.argmax(p_best_fitnesses)]
    g_best_fitness = max(p_best_fitnesses)

    for _ in range(iterations):
        for i in range(num_particles):
            # Simplified particle update
            phi = np.random.rand()
            p = particles[i]
            x_mean = np.mean(np.array([p_best_positions[i], g_best_position]), axis=0).astype(int)
            new_position = np.zeros_like(p)
            for j in range(len(p)):
                if np.random.rand() < phi:
                    new_position[j] = x_mean[j]
                else:
                    new_position[j] = p[j]

            # Ensure no duplicate features in a particle
            new_position = np.unique(new_position)
            if len(new_position) != num_features_to_select:
                remaining_features = set(range(n_features)) - set(new_position)
                new_position = np.concatenate((new_position, np.random.choice(list(remaining_features), num_features_to_select - len(new_position), replace=False)))
            particles[i] = new_position

            # Evaluate fitness
            current_fitness = simple_fitness_evaluation(particles[i], data, labels)

            # Update p_best
            if current_fitness > p_best_fitnesses[i]:
                p_best_fitnesses[i] = current_fitness
                p_best_positions[i] = particles[i]

            # Update g_best
            if current_fitness > g_best_fitness:
                g_best_fitness = current_fitness
                g_best_position = particles[i]

    return g_best_position



def simple_fitness_evaluation(feature_indices, data, labels):
    """
    Simplified fitness function (variance ratio).
    """
    # Check if feature_indices is empty using len()
    if len(feature_indices) == 0:  # Changed this line
        return 0
    normal_data = data[labels == 0][:, feature_indices]
    anomalous_data = data[labels == 1][:, feature_indices]
    if normal_data.size == 0 or anomalous_data.size == 0:
        return 0
    return np.var(anomalous_data) / (np.var(normal_data) + 1e-6)




# --- 4. Feature Selection with QPSO on Network and Contextual Features ---
anomaly_labels = (y_original != 'Benign').astype(int)  # Example: Assume 'anything not Benign' is the anomaly

num_network_features_to_select = 5  # Tune this parameter
num_contextual_features_to_select = 5  # Tune this parameter

# Filter out columns with object (string) dtype from X_network
# Use the original DataFrame (df_dropped) to select columns by type
numeric_columns_network = df_dropped.iloc[:, :n_network_features].select_dtypes(exclude=['object']).columns
X_network_numeric = X_network[:, df_dropped.columns[:n_network_features].isin(numeric_columns_network)]

print("\n--- QPSO Feature Selection for Network Features ---")
selected_network_feature_indices = qpso(
    X_network_numeric, anomaly_labels, num_features_to_select=num_network_features_to_select
)
print(f"Selected network feature indices: {selected_network_feature_indices}")

print("\n--- QPSO Feature Selection for Contextual Features ---")

# Filter out columns with object (string) dtype from X_contextual
# Use the original DataFrame (df_dropped) to select columns by type
numeric_columns_contextual = df_dropped.iloc[:, n_network_features:].select_dtypes(exclude=['object']).columns

# *** Changed here to align boolean index with X_contextual shape ***
# Get the column indices of numeric_columns_contextual in df_dropped
contextual_indices_in_df = df_dropped.columns.get_indexer(numeric_columns_contextual)
# Adjust indices to be relative to X_contextual (subtract n_network_features)
contextual_indices_in_X_contextual = contextual_indices_in_df - n_network_features

# Now use these adjusted indices to select columns from X_contextual
X_contextual_numeric = X_contextual[:, contextual_indices_in_X_contextual]

selected_contextual_feature_indices = qpso(
    X_contextual_numeric, anomaly_labels, num_features_to_select=num_contextual_features_to_select
)
print(f"Selected contextual feature indices: {selected_contextual_feature_indices}")

# Reduce the feature sets
X_network_selected = X_network_numeric[:, selected_network_feature_indices]
X_contextual_selected = X_contextual_numeric[:, selected_contextual_feature_indices]

# Combine the selected features
X_selected_context_aware = np.concatenate((X_network_selected, X_contextual_selected), axis=1)




# --- 5. Prepare Data for LSTM (Context-Aware) ---
n_samples_context_aware = X_selected_context_aware.shape[0]
n_features_context_aware = X_selected_context_aware.shape[1]
n_time_steps = 10

def create_sequences(data, labels, time_steps):
    Xs, ys = [], []
    for i in range(len(data) - time_steps):
        Xs.append(data[i:(i + time_steps)])
        ys.append(labels[i + time_steps - 1])  # Original line
    return np.array(Xs), np.array(ys)

X_sequences_context_aware, y_sequences_context_aware = create_sequences(X_selected_context_aware, y_categorical, n_time_steps)

# Split data
# Adjust y_original slicing to match the length of X_sequences_context_aware and y_sequences_context_aware
# The error was here: the length of y_original slice was incorrect
# Correct the slice to match the length of X_sequences_context_aware:
# The previous slice was off by one due to n_time_steps-1 in the stop index.
# Removing -1 will correct the slice
# The lengths of X_sequences_context_aware, y_sequences_context_aware and the third array should be the same.
# The third array in this case should be sliced to match the length of X_sequences_context_aware which is 1231401.
# Since the third array is a slice of y_original, it needs to be sliced correctly to achieve the desired length.
# The correct slice is y_original[n_time_steps - 1:len(y_original) - n_time_steps + 1] which will give the length 1231393 and hence the error.
# Changing the slice to y_original[n_time_steps - 1 : len(X_sequences_context_aware) + n_time_steps -1 ] will give the correct length of 1231401 for stratify to work correctly.
# y_original[n_time_steps - 1 : len(X_sequences_context_aware) + n_time_steps -1 ]  will give length of 1231401 same as X_sequences_context_aware
# Modified line:
X_train_context_aware, X_test_context_aware, y_train_context_aware, y_test_context_aware, y_train_original_context_aware, y_test_original_context_aware = train_test_split(
    X_sequences_context_aware, y_sequences_context_aware, y_original[n_time_steps - 1 : len(X_sequences_context_aware) + n_time_steps -1 ],
    test_size=0.2, random_state=42, stratify=y_original[n_time_steps - 1 : len(X_sequences_context_aware) + n_time_steps -1 ]
)

# Scale the data
scaler_context_aware = StandardScaler()
X_train_scaled_context_aware = scaler_context_aware.fit_transform(X_train_context_aware.reshape(-1, n_features_context_aware)).reshape(X_train_context_aware.shape)
X_test_scaled_context_aware = scaler_context_aware.transform(X_test_context_aware.reshape(-1, n_features_context_aware)).reshape(X_test_context_aware.shape)





# --- 6. Build and Train the LSTM Model (Context-Aware) ---
model_context_aware = Sequential()
model_context_aware.add(LSTM(64, activation='relu', input_shape=(n_time_steps, n_features_context_aware)))
model_context_aware.add(Dense(num_classes, activation='softmax'))

optimizer_context_aware = tf.keras.optimizers.SGD(learning_rate=0.01)
model_context_aware.compile(optimizer=optimizer_context_aware, loss='categorical_crossentropy', metrics=['accuracy'])
model_context_aware.summary()

epochs = 50
batch_size = 32

history_context_aware = model_context_aware.fit(X_train_scaled_context_aware, y_train_context_aware, epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose=1)




# --- 7. Evaluate the Model and Get Metrics (Context-Aware) ---
y_pred_prob_context_aware = model_context_aware.predict(X_test_scaled_context_aware)
y_pred_classes_context_aware = np.argmax(y_pred_prob_context_aware, axis=1)
y_pred_original_context_aware = label_encoder.inverse_transform(y_pred_classes_context_aware)

print("\n--- Evaluation Metrics (Context-Aware) ---")
accuracy_context_aware = accuracy_score(y_test_original_context_aware, y_pred_original_context_aware)
precision_context_aware = precision_score(y_test_original_context_aware, y_pred_original_context_aware, average='weighted')
recall_context_aware = recall_score(y_test_original_context_aware, y_pred_original_context_aware, average='weighted')
f1_context_aware = f1_score(y_test_original_context_aware, y_pred_original_context_aware, average='weighted')
loss_context_aware = model_context_aware.evaluate(X_test_scaled_context_aware, y_test_context_aware, verbose=0)[0]

print(f"Accuracy: {accuracy_context_aware:.4f}")
print(f"Precision (weighted): {precision_context_aware:.4f}")
print(f"Recall (weighted): {recall_context_aware:.4f}")
print(f"F1-Score (weighted): {f1_context_aware:.4f}")
print(f"Loss: {loss_context_aware:.4f}")

print("\n--- Classification Report (Context-Aware) ---")
print(classification_report(y_test_original_context_aware, y_pred_original_context_aware))




# --- 8. Plotting Training History (Context-Aware) ---
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_context_aware.history['accuracy'], label='Train Accuracy')
plt.plot(history_context_aware.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy (Context-Aware)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_context_aware.history['loss'], label='Train Loss')
plt.plot(history_context_aware.history['val_loss'], label='Validation Loss')
plt.title('Model Loss (Context-Aware)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()



# --- 9. Confusion Matrix (Context-Aware) ---
conf_matrix_context_aware = confusion_matrix(y_test_original_context_aware, y_pred_original_context_aware)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_context_aware, annot=True, fmt='d', cmap='Reds',
            xticklabels=unique_traffic_types, yticklabels=unique_traffic_types)
plt.xlabel('Predicted Traffic Type')
plt.ylabel('Actual Traffic Type')
plt.title('Confusion Matrix (LSTM - Context-Aware)')
plt.show()




# --- 10. AUC-ROC Curve (Context-Aware) ---
plt.figure(figsize=(10, 8))
lw = 2

y_test_encoded_context_aware = label_encoder.transform(y_test_original_context_aware)
y_test_onehot_context_aware = to_categorical(y_test_encoded_context_aware, num_classes=num_classes)

for i, traffic_type in enumerate(unique_traffic_types):
    fpr_context_aware, tpr_context_aware, _ = roc_curve(y_test_onehot_context_aware[:, i], y_pred_prob_context_aware[:, i])
    roc_auc_context_aware = auc(fpr_context_aware, tpr_context_aware)
    plt.plot(fpr_context_aware, tpr_context_aware, lw=lw, label=f'ROC curve (type {traffic_type}, AUC = {roc_auc_context_aware:.2f})')

plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) - One-vs-Rest (Context-Aware)')
plt.legend(loc="lower right")
plt.show()

Dataset loaded successfully from /content/drive/MyDrive/conquest-milcom/ACI-IoT-2023.csv
         Src IP  Src Port         Dst IP  Dst Port  Protocol  \
0   192.168.1.7     33344  54.230.163.60       443         6   
1   192.168.3.6     36754   91.189.91.48        80         6   
2   192.168.3.6     36754   91.189.91.48        80         6   
3   192.168.1.9      8080    192.168.1.1     40426         6   
4  192.168.1.20     40054  35.232.111.17        80         6   

             Timestamp  Flow Duration  Total Fwd Packet  Total Bwd packets  \
0  2023-01-11 09:43:40         379933                11                 11   
1  2023-01-11 09:43:51         205637                 3                  3   
2  2023-01-11 09:43:51              0                 2                  0   
3  2023-01-11 09:43:49        5030379                 1                  2   
4  2023-01-11 09:43:56          72278                 3                  4   

   Total Length of Fwd Packet  ...  Active Mean  Active S

/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:185: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


Selected network feature indices: [ 6 15 18 32 51]

--- QPSO Feature Selection for Contextual Features ---
Selected contextual feature indices: [2 3 4 1 0]


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        19,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 12)             │           780 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,980 (78.05 KB)

 Trainable params: 19,980 (78.05 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
27707/27707 ━━━━━━━━━━━━━━━━━━━━ 219s 8ms/step - accuracy: 0.7820 - loss: 0.8154 - val_accuracy: 0.8111 - val_loss: 0.6633
Epoch 2/50
27707/27707 ━━━━━━━━━━━━━━━━━━━━ 271s 8ms/step - accuracy: 0.8131 - loss: 0.6537 - val_accuracy: 0.8190 - val_loss: 0.6331
Epoch 3/50
27707/27707 ━━━━━━━━━━━━━━━━━━━━ 226s 8ms/step - accuracy: 0.8185 - loss: 0.6334 - val_accuracy: 0.8215 - val_loss: 0.6226
Epoch 4/50
27707/27707 ━━━━━━━━━━━━━━━━━━━━ 252s 8ms/step - accuracy: 0.8208 - loss: 0.6245 - val_accuracy: 0.8228 - val_loss: 0.6162
Epoch 5/50
27707/27707 ━━━━━━━━━━━━━━━━━━━━ 279s 8ms/step - accuracy: 0.8222 - loss: 0.6184 - val_accuracy: 0.8241 - val_loss: 0.6114
Epoch 6/50
27707/27707 ━━━━━━━━━━━━━━━━━━━━ 263s 8ms/step - accuracy: 0.8234 - loss: 0.6136 - val_accuracy: 0.8250 - val_loss: 0.6071
Epoch 7/50
27707/27707 ━━━━━━━━━━━━━━━━━━━━ 220s 8ms/step - accuracy: 0.8249 - loss: 0.6088 - val_accuracy: 0.8262 - val_loss: 0.6039
Epoch 8/50
27707/27707 ━━━━━━━━━━━━━━━━━━━━ 216s 8ms/step - ac